[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/06_ctr_ranking/06_proyecto_ranker_ctr.ipynb)

# Módulo 06 · Proyecto: ranker de la home de CineMatch — DCN-v2 vs LightGBM
### Proyecto — Curso «Sistemas de Recomendación: de cero a nivel Netflix»

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración estimada** | 4–6 h |
| **GPU recomendada** | L4 o A100 (T4 vale con FAST_DEV_RUN) |
| **Unidades de Colab estimadas** | ~5–10 |
| **Prerrequisitos** | Lección 06 (y 01, 02, 05) |

> En Colab: *Entorno de ejecución → Cambiar tipo de entorno → GPU*. Todo el notebook
> funciona también en CPU con `FAST_DEV_RUN = True` (más lento, mismos conceptos).

## 🏢 Contexto de negocio
Eres ML engineer del equipo de **Ranking de la Home de CineMatch**. El retrieval (módulos 04–05, y pronto el two-tower del 08) ya entrega ~500 candidatos por usuario. Hoy se ordenan por popularidad. Producto quiere un **ranker** que estime $P(\text{el usuario valora bien la película} \mid \text{se la mostramos})$ y lo necesita **calibrado**, porque el equipo de la home (módulo 07 y 13) va a mezclar esta probabilidad con otros objetivos.

El *tech lead* tiene dos dudas que resolverás con datos:
1. ¿Merece la pena un **DCN-v2** en GPU frente a un **LightGBM** (que ya sabe operar el equipo)?
2. Un compañero propone la feature "rating medio de la película" calculada con **todo** el histórico. ¿Es una buena idea?

## 📦 Dataset
**MovieLens-1M** (GroupLens): 1 M valoraciones, 6 040 usuarios con demografía (sexo, edad, ocupación, código postal), 3 706 películas con géneros y año. Lo convertimos en un problema de CTR:
- Cada valoración = una **impresión** que el usuario aceptó ver; etiqueta $y = 1$ si `rating ≥ 4` ("le gustó") y 0 si no.
- Features: usuario (id, sexo, edad, ocupación, prefijo postal), película (id, géneros multi-hot, década), contexto (hora, día de la semana) y **features de conteo point-in-time**.
- Split **temporal global** 80/10/10 (utilidades de los módulos 01/02).

## ✅ Entregables y rúbrica
| # | Entregable | Criterio |
|---|---|---|
| 1 | Pipeline de features **sin fuga** (point-in-time) | Test unitario incluido pasa |
| 2 | Demostración cuantitativa de la fuga con la feature "leaky" | Tabla offline con ambas versiones y explicación |
| 3 | LightGBM bien tuneado (early stopping en val) | AUC test ≥ baseline de popularidad + 0,05 |
| 4 | DCN-v2 en PyTorch (embeddings de todos los campos) | Logloss test dentro de ±1 % del LightGBM o mejor |
| 5 | Calibración: reliability diagram + ECE + NE para ambos | ECE < 0,02 tras calibrar (si hace falta) |
| 6 | Recomendación razonada al *tech lead* (5–10 líneas) | Considera métricas, coste, operación, latencia |

Referencias orientativas (MovieLens-1M real, `FAST_DEV_RUN=False`): AUC ≈ 0,75–0,82 y logloss ≈ 0,50–0,56 son valores típicos para este planteamiento; con datos sintéticos los números serán distintos.

In [ ]:
!pip install -q lightgbm

In [ ]:
import random, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)
if device == "cuda":
    print(torch.cuda.get_device_name(0))
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
FAST_DEV_RUN = True   # False en GPU para más épocas y LightGBM más grande

### 🧰 Utilidades mínimas (vienen de los módulos 01 y 02)

Para que este notebook sea **autocontenido en Colab** repetimos aquí, en versión
compacta, tres piezas que ya construiste antes:

1. `load_movielens(size)` — descarga MovieLens desde GroupLens (módulo 01). Si no hay
   red, genera un **MovieLens sintético** con el mismo esquema para que nada se bloquee.
2. `temporal_split(...)` — split **temporal global** (módulo 01): entrenamos con el
   pasado y evaluamos con el futuro, como en producción.
3. `recall_at_k`, `ndcg_at_k` — métricas de ranking de top-K (módulo 02).

In [ ]:
# === Utilidades mínimas (módulos 01 y 02) ===================================
import os, io, zipfile, urllib.request
import numpy as np
import pandas as pd

ML_URLS = {
    "1m": "https://files.grouplens.org/datasets/movielens/ml-1m.zip",
    "25m": "https://files.grouplens.org/datasets/movielens/ml-25m.zip",
}

def _synthetic_movielens(n_users=1500, n_items=900, seed=42):
    """MovieLens 'de juguete' con el mismo esquema que ML-1M (fallback sin red)."""
    rng = np.random.default_rng(seed)
    genres = ["Action", "Comedy", "Drama", "Thriller", "Romance", "Sci-Fi",
              "Horror", "Animation", "Documentary", "Children's"]
    G = len(genres)
    item_g = rng.dirichlet(np.ones(G) * 0.3, n_items)            # mezcla de géneros
    user_g = rng.dirichlet(np.ones(G) * 0.5, n_users)            # gustos
    pop = np.minimum(rng.zipf(1.4, n_items), 300).astype(float)  # cola larga
    activity = np.clip(rng.zipf(1.6, n_users) * 20, 20, 600).astype(int)
    rows, t0 = [], 956_703_932                                    # ~abril 2000
    for u in range(n_users):
        aff = item_g @ user_g[u]
        p = pop * (aff + 0.02) ** 2
        p /= p.sum()
        n = min(activity[u], n_items // 2)
        items = rng.choice(n_items, size=n, replace=False, p=p)
        score = 4 * (aff[items] / aff.max()) + rng.normal(0, 0.8, n)
        r = np.clip(np.round(1 + score), 1, 5).astype(int)
        start = t0 + rng.integers(0, 2 * 365 * 86400)
        ts = np.sort(start + rng.integers(0, 365 * 86400, n))
        rows.append(pd.DataFrame({"user_id": u + 1, "item_id": items + 1,
                                  "rating": r, "timestamp": ts}))
    ratings = pd.concat(rows, ignore_index=True)
    gnames = ["|".join([genres[g] for g in np.where(item_g[i] > 0.25)[0]] or
                       [genres[int(item_g[i].argmax())]]) for i in range(n_items)]
    years = rng.integers(1930, 2001, n_items)
    movies = pd.DataFrame({"item_id": np.arange(1, n_items + 1),
                           "title": [f"Película sintética {i+1} ({y})" for i, y in enumerate(years)],
                           "genres": gnames})
    users = pd.DataFrame({"user_id": np.arange(1, n_users + 1),
                          "gender": rng.choice(["M", "F"], n_users),
                          "age": rng.choice([1, 18, 25, 35, 45, 50, 56], n_users),
                          "occupation": rng.integers(0, 21, n_users),
                          "zip": [f"{z:05d}" for z in rng.integers(0, 99999, n_users)]})
    return ratings, movies, users

def load_movielens(size: str = "1m", data_dir: str = "data"):
    """Devuelve (ratings, movies, users) con columnas user_id, item_id, rating, timestamp."""
    os.makedirs(data_dir, exist_ok=True)
    try:
        zpath = os.path.join(data_dir, f"ml-{size}.zip")
        if not os.path.exists(zpath):
            print(f"Descargando MovieLens-{size} de GroupLens…")
            urllib.request.urlretrieve(ML_URLS[size], zpath)
        with zipfile.ZipFile(zpath) as z:
            if size == "1m":
                rd = lambda f, cols: pd.read_csv(io.BytesIO(z.read(f"ml-1m/{f}")), sep="::",
                                                 engine="python", names=cols, encoding="latin-1")
                ratings = rd("ratings.dat", ["user_id", "item_id", "rating", "timestamp"])
                movies = rd("movies.dat", ["item_id", "title", "genres"])
                users = rd("users.dat", ["user_id", "gender", "age", "occupation", "zip"])
            else:
                ratings = pd.read_csv(z.open("ml-25m/ratings.csv")).rename(
                    columns={"userId": "user_id", "movieId": "item_id"})
                movies = pd.read_csv(z.open("ml-25m/movies.csv")).rename(columns={"movieId": "item_id"})
                users = None
        print(f"MovieLens-{size} real: {len(ratings):,} ratings")
    except Exception as e:  # sin red / URL caída → fallback sintético
        print(f"⚠️ No se pudo descargar MovieLens ({type(e).__name__}). Uso datos SINTÉTICOS.")
        ratings, movies, users = _synthetic_movielens()
    return ratings, movies, users

def temporal_split(df: pd.DataFrame, val_frac=0.1, test_frac=0.1, ts_col="timestamp"):
    """Split temporal GLOBAL: [pasado | validación | futuro] por cuantiles de tiempo."""
    t_val, t_test = df[ts_col].quantile([1 - val_frac - test_frac, 1 - test_frac])
    train = df[df[ts_col] < t_val]
    val = df[(df[ts_col] >= t_val) & (df[ts_col] < t_test)]
    test = df[df[ts_col] >= t_test]
    return train, val, test

def recall_at_k(topk: np.ndarray, truth: list, k: int) -> float:
    """topk: [U, >=k] ids recomendados; truth: lista de sets con los relevantes."""
    vals = [len(set(row[:k]) & t) / len(t) for row, t in zip(topk, truth) if len(t) > 0]
    return float(np.mean(vals)) if vals else 0.0

def ndcg_at_k(topk: np.ndarray, truth: list, k: int) -> float:
    """NDCG@k con relevancia binaria (módulo 02)."""
    disc = 1.0 / np.log2(np.arange(2, k + 2))
    vals = []
    for row, t in zip(topk, truth):
        if not t:
            continue
        gains = np.array([1.0 if i in t else 0.0 for i in row[:k]])
        idcg = disc[: min(len(t), k)].sum()
        vals.append((gains * disc[: len(gains)]).sum() / idcg)
    return float(np.mean(vals)) if vals else 0.0

In [ ]:
# === Bucle de entrenamiento y métricas comunes para TODOS los modelos CTR =====
from sklearn.metrics import roc_auc_score, log_loss

def normalized_entropy(y, p, base_ctr=None):
    """NE de He et al. (Facebook, 2014): logloss / entropía del CTR medio de train."""
    p = np.clip(p, 1e-7, 1 - 1e-7)
    b = np.mean(y) if base_ctr is None else base_ctr
    h = -(b * np.log(b) + (1 - b) * np.log(1 - b))
    return log_loss(y, p) / h

def ctr_metrics(y, p, base_ctr=None) -> dict:
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return {"logloss": log_loss(y, p), "auc": roc_auc_score(y, p),
            "NE": normalized_entropy(y, p, base_ctr), "calib (Σp/Σy)": p.sum() / max(y.sum(), 1)}

@torch.no_grad()
def predict_proba(model, X: np.ndarray, bs: int = 65536) -> np.ndarray:
    model.eval()
    out = []
    for i in range(0, len(X), bs):
        xb = torch.as_tensor(X[i:i + bs], device=device).long()
        out.append(torch.sigmoid(model(xb)).float().cpu().numpy())
    return np.concatenate(out)

def train_ctr(model, Xtr, ytr, Xva, yva, epochs=2, bs=4096, lr=1e-3, wd=0.0, verbose=True):
    """Entrenamiento estándar CTR: Adam + BCE, evaluación por época, guarda el mejor."""
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    Xt = torch.as_tensor(Xtr, device=device)
    yt = torch.as_tensor(ytr, dtype=torch.float32, device=device)
    hist, best, best_state = [], np.inf, None
    n_steps = math.ceil(len(Xt) / bs)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(Xt), device=device)
        run = 0.0
        for s in range(n_steps):
            idx = perm[s * bs:(s + 1) * bs]
            loss = F.binary_cross_entropy_with_logits(model(Xt[idx].long()), yt[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            run += loss.item()
            if s % max(1, n_steps // 4) == 0:
                hist.append({"step": ep * n_steps + s, "train_loss": run / (s + 1)})
        pva = predict_proba(model, Xva)
        m = ctr_metrics(yva, pva)
        hist[-1].update({"val_logloss": m["logloss"], "val_auc": m["auc"]})
        if verbose:
            print(f"  época {ep+1}: train {run/n_steps:.4f} | val logloss {m['logloss']:.4f} auc {m['auc']:.4f}")
        if m["logloss"] < best:
            best = m["logloss"]
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist)

In [ ]:
ratings, movies, users = load_movielens("1m")
df = (ratings.merge(users, on="user_id", how="left").merge(movies, on="item_id", how="left")
      .sort_values("timestamp").reset_index(drop=True))
df["y"] = (df.rating >= 4).astype(np.float32)
dt = pd.to_datetime(df.timestamp, unit="s")
df["hour"], df["dow"] = dt.dt.hour, dt.dt.dayofweek
df["year"] = df.title.str.extract(r"\((\d{4})\)").astype(float).fillna(1990)
df["decade"] = (df.year // 10 * 10).astype(int)
df["zip1"] = df.zip.astype(str).str[:1]
train, val, test = temporal_split(df)
print(len(train), len(val), len(test), "| tasa de positivos:", round(df.y.mean(), 3))

## Paso 1 — Features de conteo *point-in-time* (y la versión con fuga)
Implementa `point_in_time_item_stats(df)` que, para cada fila, calcule con **solo eventos anteriores** (por timestamp, sin incluir la propia fila):
- `item_cnt_pit`: nº de valoraciones previas de la película,
- `item_pos_rate_pit`: tasa de positivos previa con suavizado bayesiano $\frac{\text{pos} + \alpha\, \bar y}{\text{cnt} + \alpha}$ ($\alpha=20$, $\bar y$ = tasa global de **train**),
- lo mismo para el usuario (`user_cnt_pit`, `user_pos_rate_pit`).

La versión con fuga (`item_pos_rate_leaky`) usa todo el dataset (ya la damos hecha).

💡 Pista: ordena por `timestamp`, usa `groupby(...).cumsum()` y resta la propia fila (`shift`).

In [ ]:
ALPHA = 20.0
PRIOR = float(train.y.mean())

def point_in_time_item_stats(df: pd.DataFrame) -> pd.DataFrame:
    # TODO: devuelve un DataFrame (mismo índice que df) con columnas
    #       item_cnt_pit, item_pos_rate_pit, user_cnt_pit, user_pos_rate_pit
    raise NotImplementedError

# Feature con FUGA (no la uses en el modelo final): tasa de positivos con TODO el histórico
df["item_pos_rate_leaky"] = df.groupby("item_id").y.transform("mean")

In [ ]:
# Test unitario (no lo modifiques): la feature de un evento NO puede depender de eventos futuros ni de sí mismo
def _test_pit(fn):
    toy = pd.DataFrame({"user_id": [1, 2, 1, 3], "item_id": [7, 7, 7, 8],
                        "timestamp": [1, 2, 3, 4], "y": [1.0, 0.0, 1.0, 1.0]})
    out = fn(toy)
    assert np.allclose(out.item_cnt_pit.values, [0, 1, 2, 0]), out
    expected_rate_3rd = (1 + 0 + ALPHA * PRIOR) / (2 + ALPHA)
    assert abs(out.item_pos_rate_pit.values[2] - expected_rate_3rd) < 1e-9
    assert abs(out.item_pos_rate_pit.values[0] - PRIOR) < 1e-9
    print("✅ point-in-time OK")
try:
    _test_pit(point_in_time_item_stats)
except NotImplementedError:
    print("⏳ Implementa point_in_time_item_stats")

## Paso 2 — Codificación de features para LightGBM y para el modelo profundo
- `CAT_FEATS` → índices enteros (vocabulario **de train**, OOV=0) y, para el deep, índices **globales** con *offsets*.
- `NUM_FEATS` → para LightGBM tal cual; para el deep, **bucketiza** en cuantiles de train (20 bins) y trátalas como categóricas.
- Géneros: multi-hot (18 columnas binarias) → para el deep, cada género es un campo binario (0/1).

In [ ]:
CAT_FEATS = ["user_id", "item_id", "gender", "age", "occupation", "zip1", "decade", "hour", "dow"]
NUM_FEATS = ["item_cnt_pit", "item_pos_rate_pit", "user_cnt_pit", "user_pos_rate_pit"]
GENRES = sorted({g for gs in movies.genres for g in gs.split("|")})

def encode(df: pd.DataFrame, train_mask: np.ndarray, num_feats: list):
    """Devuelve (X_lgb: DataFrame, X_deep: int array [N, F], field_dims)."""
    # TODO 1: añade columnas multi-hot de género 'g_<Genero>' a df
    # TODO 2: X_lgb = categóricas codificadas (vocab train, OOV=0, dtype 'category') + numéricas + géneros
    # TODO 3: X_deep = categóricas + numéricas bucketizadas (20 cuantiles de train) + géneros (0/1), con offsets globales
    raise NotImplementedError

## Paso 3 — Baseline de popularidad y LightGBM
Baseline: `item_pos_rate_pit` como score directamente. Luego entrena LightGBM con early stopping (pista: `num_leaves` 63–255, `learning_rate` 0,05, `min_data_in_leaf` ≥ 100).

In [ ]:
import lightgbm as lgb

def train_lgb(Xtr, ytr, Xva, yva):
    # TODO: entrena con early stopping y devuelve el booster
    raise NotImplementedError

## Paso 4 — DCN-v2 (reutiliza tu implementación de la lección)
Implementa `DCNv2` con capas de cruce `x_{l+1} = x0 * (W x_l + b) + x_l` y la parte deep *stacked* o *parallel*. Entrena con `train_ctr` (incluida arriba).

In [ ]:
class DCNv2(nn.Module):
    def __init__(self, field_dims, k=16, n_cross=3, hidden=(256, 256), dropout=0.1):
        super().__init__()
        # TODO
        raise NotImplementedError
    def forward(self, x):
        raise NotImplementedError

## Paso 5 — Fuga, calibración y recomendación final
1. Entrena LightGBM **con** `item_pos_rate_leaky` y compara offline (val/test) con la versión point-in-time. ¿Por qué parece mejor? ¿Qué pasaría en producción?
2. Reliability diagrams + ECE + NE para LightGBM y DCN-v2 (y una calibración isotónica en val si hace falta).
3. Escribe tu recomendación al *tech lead*.

In [ ]:
# TODO: tu análisis final aquí

---
# ⛔ SPOILER — Solución de referencia
Intenta resolverlo primero. Lo que sigue es una solución completa y ejecutable.

In [ ]:
def point_in_time_item_stats(df: pd.DataFrame) -> pd.DataFrame:
    d = df[["user_id", "item_id", "timestamp", "y"]].copy()
    d["_ord"] = np.arange(len(d))
    d = d.sort_values(["timestamp", "_ord"], kind="mergesort")
    out = pd.DataFrame(index=d.index)
    for key, pref in [("item_id", "item"), ("user_id", "user")]:
        g = d.groupby(key)
        cnt = g.cumcount().astype(float)                  # nº de eventos ANTERIORES
        pos = g.y.cumsum() - d.y                          # positivos anteriores (excluye la fila)
        out[f"{pref}_cnt_pit"] = cnt
        out[f"{pref}_pos_rate_pit"] = (pos + ALPHA * PRIOR) / (cnt + ALPHA)
    return out.loc[df.index]
# Nota: eventos con el MISMO timestamp se ordenan por orden de llegada; en producción
# se usaría además un retardo (los contadores llegan con minutos de latencia).

_test_pit(point_in_time_item_stats)
df = df.join(point_in_time_item_stats(df))
train, val, test = temporal_split(df)
tr_m = df.index.isin(train.index); va_m = df.index.isin(val.index); te_m = df.index.isin(test.index)

In [ ]:
def encode(df: pd.DataFrame, train_mask: np.ndarray, num_feats: list):
    df = df.copy()
    for g in GENRES:
        df["g_" + g] = df.genres.fillna("").str.contains(g, regex=False).astype(np.int8)
    gcols = ["g_" + g for g in GENRES]
    X_lgb = pd.DataFrame(index=df.index)
    deep_cols, field_dims = [], []
    for c in CAT_FEATS:
        vocab = pd.Index(df.loc[train_mask, c].astype(str).unique())
        codes = vocab.get_indexer(df[c].astype(str)) + 1               # -1 (OOV) → 0
        X_lgb[c] = pd.Categorical(codes)
        deep_cols.append(codes); field_dims.append(len(vocab) + 1)
    for c in num_feats:
        X_lgb[c] = df[c].values
        qs = np.unique(np.quantile(df.loc[train_mask, c], np.linspace(0, 1, 21)[1:-1]))
        deep_cols.append(np.digitize(df[c].values, qs)); field_dims.append(len(qs) + 1)
    for c in gcols:
        X_lgb[c] = df[c].values
        deep_cols.append(df[c].values.astype(int)); field_dims.append(2)
    offsets = np.cumsum([0] + field_dims[:-1])
    X_deep = (np.stack(deep_cols, 1) + offsets).astype(np.int32)
    return X_lgb, X_deep, field_dims

X_lgb, X_deep, field_dims = encode(df, tr_m, NUM_FEATS)
y = df.y.values.astype(np.float32)
print(X_lgb.shape, X_deep.shape, "tabla de embeddings:", sum(field_dims))

In [ ]:
pop_auc = roc_auc_score(y[te_m], df.item_pos_rate_pit.values[te_m])
print(f"Baseline popularidad point-in-time · AUC test = {pop_auc:.4f}")

def train_lgb(Xtr, ytr, Xva, yva):
    params = dict(objective="binary", learning_rate=0.05, num_leaves=127, min_data_in_leaf=200,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                  cat_smooth=20, min_data_per_group=100, verbose=-1, seed=seed)
    return lgb.train(params, lgb.Dataset(Xtr, ytr), num_boost_round=400 if FAST_DEV_RUN else 3000,
                     valid_sets=[lgb.Dataset(Xva, yva)], callbacks=[lgb.early_stopping(50, verbose=False)])

t0 = time.time()
gbm = train_lgb(X_lgb[tr_m], y[tr_m], X_lgb[va_m], y[va_m])
p_lgb = gbm.predict(X_lgb[te_m], num_iteration=gbm.best_iteration)
BASE = y[tr_m].mean()
R = {"Popularidad PIT": {**ctr_metrics(y[te_m], np.clip(df.item_pos_rate_pit.values[te_m], 1e-4, 1-1e-4), BASE)},
     "LightGBM": {**ctr_metrics(y[te_m], p_lgb, BASE), "seg": time.time() - t0}}
print(pd.DataFrame(R).T.round(4))

In [ ]:
class CrossLayerV2(nn.Module):
    def __init__(self, d):
        super().__init__(); self.W = nn.Linear(d, d)
    def forward(self, x0, xl):
        return x0 * self.W(xl) + xl

class DCNv2(nn.Module):
    def __init__(self, field_dims, k=16, n_cross=3, hidden=(256, 256), dropout=0.1):
        super().__init__()
        d = len(field_dims) * k
        self.emb = nn.Embedding(sum(field_dims), k); nn.init.normal_(self.emb.weight, std=0.01)
        self.cross = nn.ModuleList([CrossLayerV2(d) for _ in range(n_cross)])
        layers, a = [], d
        for h in hidden:
            layers += [nn.Linear(a, h), nn.ReLU(), nn.Dropout(dropout)]; a = h
        self.deep = nn.Sequential(*layers)
        self.head = nn.Linear(d + hidden[-1], 1)                        # estructura 'parallel'
    def forward(self, x):
        x0 = self.emb(x).flatten(1); xl = x0
        for c in self.cross:
            xl = c(x0, xl)
        return self.head(torch.cat([xl, self.deep(x0)], 1)).squeeze(1)

torch.manual_seed(seed); t0 = time.time()
dcn, hist = train_ctr(DCNv2(field_dims), X_deep[tr_m], y[tr_m], X_deep[va_m], y[va_m],
                      epochs=3 if FAST_DEV_RUN else 6, bs=2048, lr=1e-3, wd=1e-6)
p_dcn = predict_proba(dcn, X_deep[te_m])
R["DCN-v2"] = {**ctr_metrics(y[te_m], p_dcn, BASE), "seg": time.time() - t0}
pd.DataFrame(R).T.round(4)

In [ ]:
# Fuga: la misma LightGBM pero con la feature calculada con TODO el histórico
X_leak = X_lgb.copy(); X_leak["item_pos_rate_leaky"] = df.item_pos_rate_leaky.values
gbm_leak = train_lgb(X_leak[tr_m], y[tr_m], X_leak[va_m], y[va_m])
p_leak = gbm_leak.predict(X_leak[te_m], num_iteration=gbm_leak.best_iteration)
R["LightGBM + feature con FUGA"] = ctr_metrics(y[te_m], p_leak, BASE)
display(pd.DataFrame(R).T.round(4))
imp = pd.Series(gbm_leak.feature_importance("gain"), index=X_leak.columns).sort_values()[-12:]
imp.plot.barh(figsize=(6, 4), title="Importancia (gain) con la feature con fuga"); plt.show()

**Interpretación de la fuga**: `item_pos_rate_leaky` incluye la etiqueta de la propia fila y valoraciones **futuras** de la película. Offline el AUC sube, y la feature domina la importancia; en producción ese valor no existe en el momento de servir (solo conoces el pasado), así que el modelo se apoyaría en una señal que se degrada. Es el error #1 en features de conteo.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
def ece(y_, p_, bins=15):
    q = np.quantile(p_, np.linspace(0, 1, bins + 1)); b = np.clip(np.digitize(p_, q[1:-1]), 0, bins - 1)
    return sum(abs(y_[b == i].mean() - p_[b == i].mean()) * (b == i).mean() for i in range(bins) if (b == i).any())

iso = IsotonicRegression(out_of_bounds="clip").fit(predict_proba(dcn, X_deep[va_m]), y[va_m])
cands = {"LightGBM": p_lgb, "DCN-v2": p_dcn, "DCN-v2 + isotónica": iso.predict(p_dcn)}
plt.figure(figsize=(5.5, 5))
for k, p in cands.items():
    fr, mp = calibration_curve(y[te_m], p, n_bins=15, strategy="quantile")
    plt.plot(mp, fr, "o-", ms=3, label=f"{k} · ECE={ece(y[te_m], p):.4f} · NE={normalized_entropy(y[te_m], p, BASE):.3f}")
plt.plot([0, 1], [0, 1], "k--", lw=0.8); plt.legend(fontsize=7); plt.xlabel("p̂"); plt.ylabel("tasa observada")
plt.title("Reliability diagram — test temporal"); plt.show()

In [ ]:
# Coste de inferencia aproximado (CPU/GPU actual) para 500 candidatos
Xc_lgb, Xc_deep = X_lgb[te_m].iloc[:500], X_deep[te_m][:500]
t0 = time.perf_counter(); [gbm.predict(Xc_lgb, num_iteration=gbm.best_iteration) for _ in range(20)]; t_lgb = (time.perf_counter() - t0) / 20
t0 = time.perf_counter(); [predict_proba(dcn, Xc_deep) for _ in range(20)]; t_dcn = (time.perf_counter() - t0) / 20
print(f"Latencia por petición (500 candidatos): LightGBM {t_lgb*1e3:.1f} ms | DCN-v2 {t_dcn*1e3:.1f} ms ({device})")

### Recomendación modelo al *tech lead* (ejemplo de respuesta)
> En el split temporal, LightGBM y DCN-v2 quedan muy cerca en logloss/AUC sobre MovieLens-1M: con pocas features de ID y muchas agregadas, el GBDT es un rival durísimo. DCN-v2 gana cuando aumentan los IDs y las secuencias (historia del usuario), y comparte embeddings con el two-tower del módulo 08. Propongo: (1) poner en producción **LightGBM** ya (operación conocida, latencia de ~ms en CPU), con las features **point-in-time** (la versión "leaky" infla el AUC offline y fallaría online); (2) A/B de DCN-v2 cuando añadamos la historia del usuario (DIN) y multi-tarea (módulo 07); (3) monitorizar ECE/NE y el ratio Σp/Σy por segmento, y recalibrar con isotónica en cada reentreno.

## 🚀 Retos extra
1. **Criteo real**: corre la lección con `FAST_DEV_RUN=False` (5 M filas) y compara DCN-v2 *stacked* vs *parallel* vs low-rank (`rank=64`). ¿Cuánta logloss pierdes con bajo rango y cuánta memoria ganas?
2. **DIN en CineMatch**: añade la historia de las últimas 50 películas como feature secuencial al ranker de este proyecto.
3. **Mixed negative sampling y exposure**: MovieLens solo tiene ítems valorados. Añade negativos muestreados (no vistos) y estudia cómo cambia la calibración.
4. **TorchRec**: reescribe la tabla de embeddings con `EmbeddingBagCollection` y una tabla por campo.
5. **5 semillas**: repite LightGBM y DCN-v2 con 5 semillas y reporta media ± desviación. ¿Es significativa la diferencia?

## 🤔 Reflexión (producción / MLOps)
- ¿Cómo garantizarías en un **feature store** (Feast, módulo 16) que las features de conteo del entrenamiento son exactamente las que verá el modelo al servir (*training-serving skew*)?
- ¿Con qué frecuencia reentrenarías? ¿Qué métrica de monitoreo dispararía un reentreno (módulo 17)?
- Si el modelo se usa para mezclar objetivos en la home, ¿qué alerta pondrías sobre la calibración?